# Sequence training with a learnable LIF

This notebook trains a crematorium `LIF` on a synthetic task:
- input sequences are either **strong** (high-amplitude noise) or **weak** (low-amplitude noise)
- the network must learn to spike strongly for strong inputs and stay quiet for weak ones

We use the default straight-through surrogate gradient, learnable `beta` and
`threshold`, and backprop through time over `forward_sequence`.

```bash
uv run --extra cu130 pip install matplotlib
```

In [ ]:
import torch
import matplotlib.pyplot as plt

from crematorium.snn import LIF

## Task

`T` timesteps, `B` sequences per batch, `F` features. Each sequence is
labelled `1` (strong) or `0` (weak).

In [ ]:
torch.manual_seed(0)

T, B, F = 100, 32, 16


def make_batch(strong: bool):
    amp = 2.0 if strong else 0.2
    return torch.randn(T, B, F) * amp


x_strong = make_batch(True)
x_weak = make_batch(False)

## Model

Explicit mode: the caller owns the state. `learnable_beta` and
`learnable_threshold` register those params as `nn.Parameter`s; their
constraints (`clamp_unit_interval`, `clamp_positive`) are applied on the hot
path inside `_step`. The default `spike_grad` is a hard threshold forward
with a straight-through identity backward.

In [ ]:
lif = LIF(
    init_hidden=False,
    learnable_beta=True,
    learnable_threshold=True,
)
opt = torch.optim.Adam(lif.parameters(), lr=1e-2)
print(lif)
print("learnable params:", [p.shape for p in lif.parameters()])

## Training

Loss: for each sample, the mean firing rate over time and features should
match the label (`1.0` strong, `0.0` weak). Gradients flow through the
unrolled sequence scan via BPTT.

In [ ]:
losses = []

for epoch in range(60):
    opt.zero_grad()

    spk_s, _ = lif.forward_sequence(x_strong, lif.zero_state((B, F)))
    spk_w, _ = lif.forward_sequence(x_weak, lif.zero_state((B, F)))

    rate_s = spk_s.mean(dim=(0, 2))
    rate_w = spk_w.mean(dim=(0, 2))

    loss = torch.nn.functional.mse_loss(
        rate_s, torch.ones(B)
    ) + torch.nn.functional.mse_loss(rate_w, torch.zeros(B))

    loss.backward()
    opt.step()
    losses.append(loss.item())

    if epoch % 10 == 0:
        print(
            f"epoch {epoch:2d}  loss {loss.item():.4f}  "
            f"beta={lif.beta.item():.3f}  threshold={lif.threshold.item():.3f}"
        )

## Results

The trained network separates the two classes: strong inputs drive a high
mean firing rate, weak inputs stay near silence.

In [ ]:
spk_s, _ = lif.forward_sequence(x_strong, lif.zero_state((B, F)))
spk_w, _ = lif.forward_sequence(x_weak, lif.zero_state((B, F)))

print(f"mean rate strong: {spk_s.float().mean().item():.4f}")
print(f"mean rate weak:   {spk_w.float().mean().item():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].plot(losses)
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("Training loss")

axes[1].imshow(
    spk_s[:, :8, 0].detach().cpu().T,
    aspect="auto",
    cmap="Greys",
    interpolation="nearest",
)
axes[1].set_title("Strong input rasters")
axes[1].set_xlabel("time")
axes[1].set_ylabel("sample")

axes[2].imshow(
    spk_w[:, :8, 0].detach().cpu().T,
    aspect="auto",
    cmap="Greys",
    interpolation="nearest",
)
axes[2].set_title("Weak input rasters")
axes[2].set_xlabel("time")
axes[2].set_ylabel("sample")

fig.tight_layout()
plt.show()